# 02 - MPC by Hand: Constrained Finite-Horizon Control

Teaching rhythm: try -> observe -> derive/iterate -> exact solution -> limitation -> next method.

In this notebook we keep MPC manual and visible: prediction matrices, condensed cost, and explicit receding-horizon optimization.
No CasADi appears here.
        

In [ ]:
%matplotlib inline

import time
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import osqp
from scipy.linalg import solve_discrete_are, block_diag

np.set_printoptions(precision=4, suppress=True)
        

## 1) Recap from 01: LQR and clipping limitation

We use the same double-integrator benchmark and compare unconstrained LQR against saturated LQR under input limits.
        

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.diag([4.0, 1.0])
R = np.array([[0.2]])

P_inf = solve_discrete_are(A, B, Q, R)
K_inf = np.linalg.solve(R + B.T @ P_inf @ B, B.T @ P_inf @ A)

x0_demo = np.array([4.0, 1.2])
steps_demo = 28
u_max = 0.8

X_lqr = np.zeros((steps_demo + 1, 2))
U_lqr = np.zeros(steps_demo)
X_sat = np.zeros((steps_demo + 1, 2))
U_sat = np.zeros(steps_demo)

X_lqr[0] = x0_demo
X_sat[0] = x0_demo

J_lqr = 0.0
J_sat = 0.0

for k in range(steps_demo):
    u_lqr = float(-(K_inf @ X_lqr[k]).item())
    u_sat = np.clip(float(-(K_inf @ X_sat[k]).item()), -u_max, u_max)

    U_lqr[k] = u_lqr
    U_sat[k] = u_sat

    J_lqr += X_lqr[k] @ Q @ X_lqr[k] + u_lqr * R[0, 0] * u_lqr
    J_sat += X_sat[k] @ Q @ X_sat[k] + u_sat * R[0, 0] * u_sat

    X_lqr[k + 1] = A @ X_lqr[k] + B[:, 0] * u_lqr
    X_sat[k + 1] = A @ X_sat[k] + B[:, 0] * u_sat

print(f'LQR finite cost surrogate:       {J_lqr:.4f}')
print(f'Saturated LQR cost surrogate:    {J_sat:.4f}')

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(X_lqr[:, 0], label='LQR')
ax[0].plot(X_sat[:, 0], '--', label='sat LQR')
ax[0].set_title('Position')
ax[0].set_xlabel('k')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(X_lqr[:, 1], label='LQR')
ax[1].plot(X_sat[:, 1], '--', label='sat LQR')
ax[1].set_title('Velocity')
ax[1].set_xlabel('k')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

ax[2].step(range(steps_demo), U_lqr, where='post', label='LQR')
ax[2].step(range(steps_demo), U_sat, where='post', linestyle='--', label='sat LQR')
ax[2].axhline(u_max, color='r', linestyle=':', linewidth=1.0, label='input bounds')
ax[2].axhline(-u_max, color='r', linestyle=':', linewidth=1.0)
ax[2].set_title('Input')
ax[2].set_xlabel('k')
ax[2].grid(True, alpha=0.3)
ax[2].legend()

plt.tight_layout()
plt.show()
        

### Observation

Clipping changes behavior, but it is not a principled constrained optimum.
The constrained infinite-horizon problem is the right target, but not tractable online.
MPC approximates it with finite horizon plus terminal ingredients.
        

## 2) Scalar microscope: finite horizon and terminal cost

Use x[k+1] = x[k] + u[k], q = 1, r = 1.
We examine N = 1, N = 2, and terminal-cost choices.
        

In [ ]:
a = 1.0
b = 1.0
q = 1.0
r = 1.0

# Positive scalar Riccati root for a = b = q = r = 1.
p_inf_scalar = 0.5 * (q + np.sqrt(q ** 2 + 4.0 * q * r))
K_inf_scalar = p_inf_scalar / (r + p_inf_scalar)

print(f'p_inf (scalar) = {p_inf_scalar:.10f}')
print(f'K_inf (scalar) = {K_inf_scalar:.10f}')
        

### Case A: N = 1, V_f = 0

For N = 1 and no terminal cost, x0^2 is constant with respect to u0,
so the optimizer chooses u0 mainly from input penalty and appears shortsighted.
        

In [ ]:
x0_case = 4.0
u_grid = np.linspace(-5.0, 2.0, 400)
J_grid = q * x0_case ** 2 + r * u_grid ** 2
u_best = float(u_grid[np.argmin(J_grid)])

x0_scan = np.linspace(-5.0, 5.0, 81)
u0_scan = np.zeros_like(x0_scan)

for i, x0_val in enumerate(x0_scan):
    J_local = q * x0_val ** 2 + r * u_grid ** 2
    u0_scan[i] = float(u_grid[np.argmin(J_local)])

print(f'For x0 = {x0_case:.1f}, N=1, V_f=0 gives u0* = {u_best:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(u_grid, J_grid)
ax[0].axvline(u_best, color='C1', linestyle='--', label='u0*')
ax[0].set_title('Case A: cost versus u0')
ax[0].set_xlabel('u0')
ax[0].set_ylabel('J')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(x0_scan, u0_scan)
ax[1].set_title('Case A: first action map u0*(x0)')
ax[1].set_xlabel('x0')
ax[1].set_ylabel('u0*')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
        

### Case B and C: N = 2 with terminal weight alpha

We build Phi and Gamma explicitly for the scalar system and vary alpha in
V_f = alpha * x_N^2.
        

In [ ]:
N_scalar = 2

# Explicit scalar prediction matrices for X = [x1, x2].
Phi_s = np.zeros((N_scalar, 1))
Gamma_s = np.zeros((N_scalar, N_scalar))

for i in range(1, N_scalar + 1):
    Phi_s[i - 1, 0] = a ** i
    for j in range(i):
        Gamma_s[i - 1, j] = (a ** (i - 1 - j)) * b

print('Phi_s =')
print(Phi_s)
print('Gamma_s =')
print(Gamma_s)

x0_grid = np.linspace(-5.0, 5.0, 121)
alpha_values = [0.0, 1.0, 5.0, p_inf_scalar]
u0_curves = {}

for alpha in alpha_values:
    Qbar_s = np.diag([q, alpha])
    Rbar_s = np.diag([r, r])
    H_s = Gamma_s.T @ Qbar_s @ Gamma_s + Rbar_s

    u0_values = np.zeros_like(x0_grid)
    for idx, x0_val in enumerate(x0_grid):
        h_s = Gamma_s.T @ Qbar_s @ (Phi_s[:, 0] * x0_val)
        U_star = -np.linalg.solve(H_s, h_s)
        u0_values[idx] = U_star[0]
    u0_curves[alpha] = u0_values

fig, ax = plt.subplots(figsize=(7, 4))
for alpha in alpha_values:
    label = 'alpha = P_inf' if np.isclose(alpha, p_inf_scalar) else f'alpha = {alpha:g}'
    ax.plot(x0_grid, u0_curves[alpha], label=label)
ax.axline((0, 0), slope=-K_inf_scalar, color='k', linestyle='--', label='LQR law -K_inf x0')
ax.set_title('Case C: first action map versus terminal weight')
ax.set_xlabel('x0')
ax.set_ylabel('u0*')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
        

### Case D: alpha = P_inf recovers scalar LQR (unconstrained)
        

In [ ]:
def scalar_u0_unconstrained(x0_val, N, alpha):
    Phi = np.zeros((N, 1))
    Gamma = np.zeros((N, N))

    for i in range(1, N + 1):
        Phi[i - 1, 0] = a ** i
        for j in range(i):
            Gamma[i - 1, j] = (a ** (i - 1 - j)) * b

    Qbar = np.diag([q] * (N - 1) + [alpha])
    Rbar = np.diag([r] * N)
    H = Gamma.T @ Qbar @ Gamma + Rbar
    h = Gamma.T @ Qbar @ (Phi[:, 0] * x0_val)
    U_star = -np.linalg.solve(H, h)
    return float(U_star[0])

for x0_test in [-4.0, -2.0, -0.5, 0.5, 2.0, 4.0]:
    u_mpc = scalar_u0_unconstrained(x0_test, N=6, alpha=p_inf_scalar)
    u_lqr = -K_inf_scalar * x0_test
    print(f'x0={x0_test:+4.1f} -> u0_mpc={u_mpc:+.6f}, u_lqr={u_lqr:+.6f}')
    assert np.isclose(u_mpc, u_lqr, atol=1e-10)

print('Check passed: unconstrained MPC with terminal P_inf matches scalar LQR first action.')
        

## 3) One-step constrained scalar problem

When unconstrained optimum violates |u| <= u_max, the active boundary becomes optimal.
This is the small-scale picture behind inequality-constrained QPs.
        

In [ ]:
x0_con = 4.0
r_con = 0.1
alpha_con = 1.0
u_max_con = 1.0

u_axis = np.linspace(-5.0, 2.0, 400)
J_axis = r_con * u_axis ** 2 + alpha_con * (x0_con + u_axis) ** 2

u_star_uncon = -alpha_con * x0_con / (r_con + alpha_con)
u_star_con = np.clip(u_star_uncon, -u_max_con, u_max_con)

print(f'Unconstrained optimum u* = {u_star_uncon:.4f}')
print(f'Constrained optimum u*   = {u_star_con:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(u_axis, J_axis, label='J(u)')
plt.axvline(u_star_uncon, color='C1', linestyle='--', label='unconstrained optimum')
plt.axvline(u_star_con, color='C2', linestyle='--', label='constrained optimum')
plt.axvline(u_max_con, color='k', linestyle=':', linewidth=1.0)
plt.axvline(-u_max_con, color='k', linestyle=':', linewidth=1.0, label='feasible interval')
plt.title('One-step constrained scalar optimization')
plt.xlabel('u')
plt.ylabel('J(u)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
        

## 4) Double integrator MPC: build Phi and Gamma manually

Now we derive the condensed formulation by hand for the matrix benchmark.
Conventions:
- X = [x1, x2, ..., xN]
- U = [u0, u1, ..., u_{N-1}]
        

In [ ]:
N = 10
x0 = np.array([3.2, 1.2])
P_f = P_inf.copy()

x_max = 4.8
v_max = 2.5
u_max = 0.8

nx = A.shape[0]
nu = B.shape[1]

Phi = np.zeros((nx * N, nx))
Gamma = np.zeros((nx * N, nu * N))

for i in range(1, N + 1):
    Phi[(i - 1) * nx : i * nx, :] = np.linalg.matrix_power(A, i)
    for j in range(i):
        Gamma[(i - 1) * nx : i * nx, j * nu : (j + 1) * nu] = np.linalg.matrix_power(A, i - 1 - j) @ B

print('Phi shape:', Phi.shape)
print('Gamma shape:', Gamma.shape)

N_small = 3
Phi_small = np.zeros((nx * N_small, nx))
Gamma_small = np.zeros((nx * N_small, nu * N_small))
for i in range(1, N_small + 1):
    Phi_small[(i - 1) * nx : i * nx, :] = np.linalg.matrix_power(A, i)
    for j in range(i):
        Gamma_small[(i - 1) * nx : i * nx, j * nu : (j + 1) * nu] = np.linalg.matrix_power(A, i - 1 - j) @ B

print('Phi for N=3:')
print(Phi_small)
print('Gamma for N=3:')
print(Gamma_small)
        

## 5) Condensed cost with explicit terminal convention

Lecture form:
J(U) = U.T H U + 2 h.T U + constant

with
H = Gamma.T Qbar Gamma + Rbar
h = Gamma.T Qbar Phi x0

For X = [x1, ..., xN], we use
Qbar = diag(Q, ..., Q, P_f)
with Q on x1..x_{N-1} and P_f on xN.

Solver form (0.5 U.T P U + q.T U):
P_solver = 2 H
q_solver = 2 h
        

In [ ]:
Q_blocks = [Q for _ in range(N - 1)] + [P_f]
R_blocks = [R for _ in range(N)]
Qbar = block_diag(*Q_blocks)
Rbar = block_diag(*R_blocks)

H = Gamma.T @ Qbar @ Gamma + Rbar
h = Gamma.T @ Qbar @ (Phi @ x0)

P_solver = 2.0 * H
q_solver = 2.0 * h

print('Qbar terminal block (should equal P_f):')
print(Qbar[-nx:, -nx:])
print('P_f:')
print(P_f)
print('H shape:', H.shape)
print('h shape:', h.shape)

assert np.allclose(Qbar[-nx:, -nx:], P_f)
print('Check passed: Qbar uses terminal P_f without adding Q at xN.')
        

### OSQP helper for inequality QPs

We solve problems of the form
min 0.5 U.T P U + q.T U subject to A_ineq U <= b_ineq.
        

In [ ]:
def solve_qp_osqp(P, q, A_ineq, b_ineq):
    P_sym = 0.5 * (P + P.T)
    P_csc = sp.csc_matrix(P_sym)
    A_csc = sp.csc_matrix(A_ineq)

    l = -np.inf * np.ones(A_ineq.shape[0])
    u = b_ineq

    prob = osqp.OSQP()
    prob.setup(
        P=P_csc,
        q=q,
        A=A_csc,
        l=l,
        u=u,
        verbose=False,
        polish=True,
        eps_abs=1e-7,
        eps_rel=1e-7,
        max_iter=20000,
    )
    res = prob.solve()

    if res.info.status_val not in (1, 2):
        raise RuntimeError(f'OSQP failed: {res.info.status}')

    return res.x, res.info
        

## 6) Unconstrained condensed MPC and LQR check
        

In [ ]:
U_unconstrained = -np.linalg.solve(H, h)
u0_mpc_unconstrained = float(U_unconstrained[0])
u0_lqr = float(-(K_inf @ x0).item())

print(f'u0 from unconstrained condensed MPC = {u0_mpc_unconstrained:+.10f}')
print(f'u0 from LQR law                    = {u0_lqr:+.10f}')

assert np.isclose(u0_mpc_unconstrained, u0_lqr, atol=1e-10)
print('Check passed: unconstrained MPC with P_f=P_inf matches LQR first action.')
        

## 7) Add input constraints in matrix form

|u_k| <= u_max is written as
G_u U <= g_u,
with G_u = [I; -I].
        

In [ ]:
G_u = np.vstack([np.eye(nu * N), -np.eye(nu * N)])
g_u = u_max * np.ones(2 * nu * N)

U_input, info_input = solve_qp_osqp(P_solver, q_solver, G_u, g_u)

print(f'Input-constrained first action u0 = {U_input[0]:+.6f}')
print(f'Max abs input in plan             = {np.max(np.abs(U_input)):.6f}')
print('OSQP status:', info_input.status)
        

## 8) Add state constraints

We enforce
|position_k| <= x_max,
|velocity_k| <= v_max,
for predicted states X.

With F_x X <= f_x and X = Phi x0 + Gamma U, this becomes
F_x Gamma U <= f_x - F_x Phi x0.
        

In [ ]:
F_single = np.array([
    [1.0, 0.0],
    [-1.0, 0.0],
    [0.0, 1.0],
    [0.0, -1.0],
])
f_single = np.array([x_max, x_max, v_max, v_max])

F_x = block_diag(*([F_single] * N))
f_x = np.tile(f_single, N)

A_state = F_x @ Gamma
b_state = f_x - F_x @ (Phi @ x0)

A_ineq = np.vstack([G_u, A_state])
b_ineq = np.concatenate([g_u, b_state])

U_state, info_state = solve_qp_osqp(P_solver, q_solver, A_ineq, b_ineq)
X_state_plan = Phi @ x0 + Gamma @ U_state

print(f'State-constrained first action u0 = {U_state[0]:+.6f}')
print('Max predicted position magnitude   =', np.max(np.abs(X_state_plan[0::nx])))
print('Max predicted velocity magnitude   =', np.max(np.abs(X_state_plan[1::nx])))
print('OSQP status:', info_state.status)
        

## 9) Receding horizon simulation (explicit loop first)

At each step:
1. solve constrained QP,
2. apply only the first input,
3. simulate one step,
4. repeat.
        

In [ ]:
sim_steps = 24
x_init = np.array([3.0, 1.1])

X_mpc = np.zeros((sim_steps + 1, nx))
U_mpc = np.zeros(sim_steps)
X_mpc[0] = x_init

solve_times_ms = []

for k in range(sim_steps):
    xk = X_mpc[k]
    h_k = Gamma.T @ Qbar @ (Phi @ xk)
    q_k = 2.0 * h_k

    b_state_k = f_x - F_x @ (Phi @ xk)
    A_ineq_k = np.vstack([G_u, A_state])
    b_ineq_k = np.concatenate([g_u, b_state_k])

    t0 = time.perf_counter()
    U_plan, info = solve_qp_osqp(P_solver, q_k, A_ineq_k, b_ineq_k)
    solve_times_ms.append(1000.0 * (time.perf_counter() - t0))

    u0 = float(U_plan[0])
    U_mpc[k] = u0
    X_mpc[k + 1] = A @ xk + B[:, 0] * u0

assert np.all(np.abs(U_mpc) <= u_max + 1e-8)
print('Average QP solve time (ms):', np.mean(solve_times_ms))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(X_mpc[:, 0], 'o-', label='position')
ax[0].axhline(x_max, color='r', linestyle=':')
ax[0].axhline(-x_max, color='r', linestyle=':')
ax[0].set_title('Constrained MPC position')
ax[0].set_xlabel('k')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(X_mpc[:, 1], 'o-', label='velocity')
ax[1].axhline(v_max, color='r', linestyle=':')
ax[1].axhline(-v_max, color='r', linestyle=':')
ax[1].set_title('Constrained MPC velocity')
ax[1].set_xlabel('k')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

ax[2].step(range(sim_steps), U_mpc, where='post', label='u')
ax[2].axhline(u_max, color='r', linestyle=':', label='u bounds')
ax[2].axhline(-u_max, color='r', linestyle=':')
ax[2].set_title('Constrained MPC input')
ax[2].set_xlabel('k')
ax[2].grid(True, alpha=0.3)
ax[2].legend()

plt.tight_layout()
plt.show()
        

## 10) Compare controllers: LQR, saturated LQR, constrained MPC
        

In [ ]:
X_lqr_cmp = np.zeros((sim_steps + 1, nx))
U_lqr_cmp = np.zeros(sim_steps)
X_sat_cmp = np.zeros((sim_steps + 1, nx))
U_sat_cmp = np.zeros(sim_steps)

X_lqr_cmp[0] = x_init
X_sat_cmp[0] = x_init

for k in range(sim_steps):
    u_lqr = float(-(K_inf @ X_lqr_cmp[k]).item())
    u_sat = np.clip(float(-(K_inf @ X_sat_cmp[k]).item()), -u_max, u_max)

    U_lqr_cmp[k] = u_lqr
    U_sat_cmp[k] = u_sat

    X_lqr_cmp[k + 1] = A @ X_lqr_cmp[k] + B[:, 0] * u_lqr
    X_sat_cmp[k + 1] = A @ X_sat_cmp[k] + B[:, 0] * u_sat

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(X_lqr_cmp[:, 0], label='LQR')
ax[0].plot(X_sat_cmp[:, 0], '--', label='sat LQR')
ax[0].plot(X_mpc[:, 0], '-.', label='MPC')
ax[0].axhline(x_max, color='k', linestyle=':', linewidth=0.8)
ax[0].axhline(-x_max, color='k', linestyle=':', linewidth=0.8)
ax[0].set_title('Position comparison')
ax[0].set_xlabel('k')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(X_lqr_cmp[:, 1], label='LQR')
ax[1].plot(X_sat_cmp[:, 1], '--', label='sat LQR')
ax[1].plot(X_mpc[:, 1], '-.', label='MPC')
ax[1].axhline(v_max, color='k', linestyle=':', linewidth=0.8)
ax[1].axhline(-v_max, color='k', linestyle=':', linewidth=0.8)
ax[1].set_title('Velocity comparison')
ax[1].set_xlabel('k')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

ax[2].step(range(sim_steps), U_lqr_cmp, where='post', label='LQR')
ax[2].step(range(sim_steps), U_sat_cmp, where='post', linestyle='--', label='sat LQR')
ax[2].step(range(sim_steps), U_mpc, where='post', linestyle='-.', label='MPC')
ax[2].axhline(u_max, color='k', linestyle=':', linewidth=0.8)
ax[2].axhline(-u_max, color='k', linestyle=':', linewidth=0.8)
ax[2].set_title('Input comparison')
ax[2].set_xlabel('k')
ax[2].grid(True, alpha=0.3)
ax[2].legend()

plt.tight_layout()
plt.show()
        

## 11) Chapter 3.8 tuning: approximation engineering

We keep the same MPC implementation and only change design parameters:
- horizon N,
- Q,
- R.
        

In [ ]:
def build_prediction_matrices(A, B, N):
    nx = A.shape[0]
    nu = B.shape[1]
    Phi = np.zeros((nx * N, nx))
    Gamma = np.zeros((nx * N, nu * N))
    for i in range(1, N + 1):
        Phi[(i - 1) * nx : i * nx, :] = np.linalg.matrix_power(A, i)
        for j in range(i):
            Gamma[(i - 1) * nx : i * nx, j * nu : (j + 1) * nu] = np.linalg.matrix_power(A, i - 1 - j) @ B
    return Phi, Gamma


def run_constrained_mpc_case(x0, A, B, Q, R, P_f, N, steps, x_max, v_max, u_max):
    nx = A.shape[0]
    nu = B.shape[1]

    Phi, Gamma = build_prediction_matrices(A, B, N)

    Qbar = block_diag(*([Q for _ in range(N - 1)] + [P_f]))
    Rbar = block_diag(*([R for _ in range(N)]))

    H = Gamma.T @ Qbar @ Gamma + Rbar
    P_solver = 2.0 * H

    G_u = np.vstack([np.eye(nu * N), -np.eye(nu * N)])
    g_u = u_max * np.ones(2 * nu * N)

    F_single = np.array([[1.0, 0.0], [-1.0, 0.0], [0.0, 1.0], [0.0, -1.0]])
    f_single = np.array([x_max, x_max, v_max, v_max])
    F_x = block_diag(*([F_single] * N))
    f_x = np.tile(f_single, N)
    A_state = F_x @ Gamma

    X = np.zeros((steps + 1, nx))
    U = np.zeros(steps)
    X[0] = x0

    solve_times = []

    for k in range(steps):
        xk = X[k]
        h_k = Gamma.T @ Qbar @ (Phi @ xk)
        q_k = 2.0 * h_k

        b_state = f_x - F_x @ (Phi @ xk)

        A_ineq = np.vstack([G_u, A_state])
        b_ineq = np.concatenate([g_u, b_state])

        t0 = time.perf_counter()
        U_plan, info = solve_qp_osqp(P_solver, q_k, A_ineq, b_ineq)
        solve_times.append(1000.0 * (time.perf_counter() - t0))

        U[k] = float(U_plan[0])
        X[k + 1] = A @ xk + B[:, 0] * U[k]

    return X, U, float(np.mean(solve_times))


x_tune = np.array([3.0, 1.1])
steps_tune = 18

cases = [
    ('baseline', 10, np.diag([4.0, 1.0]), np.array([[0.2]])),
    ('longer horizon', 16, np.diag([4.0, 1.0]), np.array([[0.2]])),
    ('more aggressive Q', 10, np.diag([9.0, 2.0]), np.array([[0.2]])),
    ('larger R', 10, np.diag([4.0, 1.0]), np.array([[1.0]])),
]

runs = {}
for name, N_case, Q_case, R_case in cases:
    X_case, U_case, avg_ms = run_constrained_mpc_case(
        x_tune, A, B, Q_case, R_case, P_inf, N_case, steps_tune, x_max, v_max, u_max
    )
    runs[name] = {
        'X': X_case,
        'U': U_case,
        'N': N_case,
        'Q': Q_case,
        'R': R_case,
        'avg_ms': avg_ms,
    }

for name, data in runs.items():
    print(f"{name:16s} | N={data['N']:2d} | avg solve {data['avg_ms']:.2f} ms")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, data in runs.items():
    ax[0].plot(data['X'][:, 0], label=name)
ax[0].set_title('Tuning effect on position')
ax[0].set_xlabel('k')
ax[0].axhline(x_max, color='k', linestyle=':', linewidth=0.8)
ax[0].axhline(-x_max, color='k', linestyle=':', linewidth=0.8)
ax[0].grid(True, alpha=0.3)
ax[0].legend()

for name, data in runs.items():
    ax[1].step(range(steps_tune), data['U'], where='post', label=name)
ax[1].set_title('Tuning effect on input')
ax[1].set_xlabel('k')
ax[1].axhline(u_max, color='k', linestyle=':', linewidth=0.8)
ax[1].axhline(-u_max, color='k', linestyle=':', linewidth=0.8)
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()
        

## Takeaway

MPC tuning is approximation engineering:
- longer horizons improve look-ahead but cost computation,
- larger Q pushes state regulation harder,
- larger R reduces control effort.

The implementation stayed the same; only design parameters changed.
        